In [ ]:
import matplotlib.pyplot as plt
import pandas
import scipy.stats
import seaborn

pandas.set_option("future.no_silent_downcasting", True)

# Data exploration: [AMES dataset](http://jse.amstat.org/v19n3/decock.pdf)

## Introduction

In this notebook, we will explore a dataset that we will use again later on during the formation to train models.

This dataset is a good playground because it contains all kinds of variables, each with its set of challenges.

Today, we will follow the following steps:

1. obtain data
2. clean data
3. explore data

## Problem statement

The end goal for our study is to be able to predict prices for the housing market in Iowa, in the US.

In order to have a correctly stated problem, it's required to formulate from the get go what are the expected outputs and how we will evaluate them.

- *Define the model output given the end goal.*

- *Suggest one or several ways to evaluate the quality of the models we will develop later on.*

- *What would be the issue if we delayed the definition of those output and evaluation elements?*

### Solution

- the model output is the house prices
- the difference between what was predicted and the reality, probably using absolute value or squaring
- the introduction of a statistical bias: we could pick the best-performing metric for the model we're evaluating

## Fetching the Data

Once the data's git repository is cloned, the data is in the `ames/` directory. It contains in particular a `train.csv` file you can use as training and validation set, and a `test.csv` file that simulates production use: the targets aren't available.

Tips:

- [`pandas.read_csv`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html) will be useful. The index column of the data is named `Id` in the CSV, so you can pass the `index_col="Id"` argument to the method along with the CSV path.

- Feel free to check [the documentation of pandas `DataFrame`s](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html).

Exercise:

- *Display the names of the dataframes' columns, along with the dimensions of the data (number of rows, number of columns)*.

- *A description of the CSV's variables is given in the data_description.txt file. Display it in a cell (or download it)*.

In [ ]:
from google.colab import files

!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s ames
!ls -l ames/
files.download("ames/data_description.txt")

In [ ]:
# train_df = pandas.???
# test_df = pandas.???

### Solution

In [ ]:
train_df = pandas.read_csv("ames/train.csv", index_col=["Id"])
test_df = pandas.read_csv("ames/test.csv", index_col="Id")

print(f"Columns: {', '.join(train_df.columns)}")
print(f"Shape of the train DataFrame: {train_df.shape}")
print(f"Shape of the test DataFrame: {test_df.shape}")

In [ ]:
train_df

## Extracting variables

*Create the following variables:*

- *`train_X` that contains all the columns of `train_df` except `SalePrice`. You can use indexing or [`DataFrame.drop`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.drop.html?highlight=drop#pandas.DataFrame.drop)*
- *`test_X` that contains all the columns of `test_df`*
- *`train_y` that contains the `SalePrice` column of `train_df`*

*Also create the `all_X` variable, using [`pandas.concat`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.concat.html?highlight=concat#pandas.concat), to concatenate `train_X` and `test_X`. That will allow us to have an easier time manipulating the two dataframes.*

*__Warning, using `test_X` (and therefore `all_X`) can introduce a statistical bias. You should only use `train_X` when exploring data or to compute a value!__*

In [ ]:
# train_X = ???
# train_y = ???
# test_X = ???
# all_X = ???

### Solution

In [ ]:
train_X = train_df.drop(columns="SalePrice")
train_y = train_df["SalePrice"]
test_X = test_df
all_X = pandas.concat([train_X, test_X])

print("Shape of the training data:", train_X.shape, train_y.shape)
print("Shape of the testing data:", test_X.shape)
print("Shape of the concatenation: ", all_X.shape)

In [ ]:
train_df[["SalePrice", "MSSubClass"]]

## Data Cleaning

This step, together with exploration, involves (at least) 4 steps:

- handling missing values
- preprocessing (text, image, …)
- standardization
- transformation

Here, we won't need specific preprocessing, since the data is mainly categorical or continuous, without text or image data.

### Missing Values

Format differences, data gathering conditions and many other factors lead to many missing values in most datasets.

In this dataset, many columns take the value `NA` as an expected value. We'll still treat these values as real missing values since, as we'll see, there are many columns for which we can do better than leaving the `NA` category in place.

It's often useful to quantify what's missing and to make up for it when it makes sense. For instance, note that to use `sklearn`, no value can be missing for almost all models to work.

- *With the [`pandas.DataFrame.isnull`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.isnull.html) function, find the number of missing elements in the training set.*
- *Adapt this process to compute the percentage of missing elements per column.*
- *Sort this result to display first the columns missing the most elements. You can use [`DataFrame.sort_values`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.sort_values.html?highlight=sort_values#pandas.DataFrame.sort_values)*
- *Also apply this process to the test set (other new columns have missing values).*
- *Display all the columns containing at least one missing value (train and test sets combined).*

In [ ]:
# Total number of missing values in train_X
# Your code here

In [ ]:
# Percentage of missing values by column in train_X
# Your code here

# Sorted
# Your code here

In [ ]:
# Total number of missing values in test_X
# Your code here

In [ ]:
# Percentage of missing values by column in test_X
# Your code here

# Sorted
# Your code here

In [ ]:
# All the columns with missing values (in either train_X or test_X)
# Your code here

#### Solution

In [ ]:
print("Total number of missing values in train_X:", train_X.isnull().sum().sum())

In [ ]:
type(train_X.isnull().sum())

In [ ]:
def columns_with_missing_values(df: pandas.DataFrame) -> pandas.Series:
  df_na: pandas.Dataframe = (df.isnull().sum() / df.shape[0]) * 100
  df_na = df_na[df_na > 0]
  return df_na.sort_values(ascending=False)


def display_percentages_series(s: pandas.Series) -> None:
  print("\n".join([f"{name:>20.20} {value:5.2f}%" for name, value in s.items()]))


print("Percentage of missing values per feature in the train set:")
missing_train = columns_with_missing_values(train_X)
display_percentages_series(missing_train)

print("Percentage of missing values per feature in the test set:")
missing_test = columns_with_missing_values(test_X)
display_percentages_series(missing_test)

In [ ]:
[f"{name:20.20} {value:5.2f}%" for name, value in missing_train.items()]

In [ ]:
print("Total number of missing values in test_X:", test_X.isnull().sum().sum())

In [ ]:
print("Columns with missing values in either train_X or test_X:")
print(", ".join(set(missing_train.index).union(missing_test.index)))
print()

print("Columns with missing values only in train_X:")
print(", ".join(set(missing_train.index).difference(missing_test.index)))
print()

print("Columns with missing values only in test_X:")
print(", ".join(set(missing_test.index).difference(missing_train.index)))

### Handling Missing Data

There's no universal approach to filling missing values. Each variable must be handled case by case.

*Based on the documentation in the `ames/data_description.txt` file, use groups `cols_1` to `cols_4` to apply 4 different ways of handling missing values. All of them involve the [`DataFrame.fillna`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.fillna.html) function, which fills missing values with the arguments given to it.*

*Each group of columns will be associated with a method for replacing missing data*

*You can, for instance, use:*

- *the mean, the median or 0 for a real variable*
- *the mode for a categorical variable (the most frequent value)*
- *an "NA" class, especially if it already exists*
- *…*

In [ ]:
# Your code here
cols_1 = ["Alley", "featureX", "..."]
cols_2 = ["SaleType", "featureY", "..."]
# etc…

# all_X[cols_1] = all_X[cols_1].fillna("defaultValue")

#### Solution

In [ ]:
# Fill with the mean
cols_1 = ["LotFrontage", "GarageYrBlt"]
all_X.fillna(train_X[cols_1].mean(), inplace=True)

# Fill with the mode
cols_2 = [
  "MSZoning",
  "Electrical",
  "KitchenQual",
  "Exterior1st",
  "Exterior2nd",
  "SaleType",
  "Utilities",
]
all_X.fillna(train_X[cols_2].mode().iloc[0, :], inplace=True)

# Fill with 0
cols_4 = [
  "GarageArea",
  "GarageCars",
  "BsmtFinSF1",
  "BsmtFinSF2",
  "BsmtFullBath",
  "BsmtHalfBath",
  "BsmtUnfSF",
  "MasVnrArea",
  "TotalBsmtSF",
]
all_X[cols_4] = all_X[cols_4].fillna(0)

# A specific fill
cols_5 = ["Functional"]
all_X[cols_5] = all_X[cols_5].fillna("Typ")

# Give all other NAs the string value NA, which will be a category
all_X = all_X.fillna("NA")

# Check that we didn't forget anything
print(all_X.isnull().sum().sum())

### Categorical Variables Encoded as Numbers

Some categories, such as `MSSubClass`, are encoded with numerical codes expressing categories (see `data_description.txt`). If we do nothing, this variable will be treated as numerical, even though the categories it describes aren't even ordinal.

In [ ]:
# To achieve that, we just have to cast the feature as a string column
cols_numerical2label = ["MSSubClass"]
all_X[cols_numerical2label] = all_X[cols_numerical2label].astype(str)

### Ordinal variables

In order not to lose information for our ordinal variables, we will have to use label encoding as seen during the presentation to transform them for later use.

- *Use label encoding for the `BsmtCond` and `FireplaceQu` variables. You can use the [`DataFrame.replace`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.replace.html) function.*

In [ ]:
# all_X=all_X.replace(???)

#### Solution

In [ ]:
# Build the dictionary of dictionaries used to replace nominal variables with integers
# Key   : "Variable name"
# Value :  Replacement dictionary of the variable's values
replace_mapping = dict(
  Alley=dict(NA=0, Grvl=1, Pave=2),
  BsmtExposure=dict(NA=0, No=1, Mn=2, Av=3, Gd=4),
  BsmtFinType1=dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6),
  BsmtFinType2=dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6),
  Functional=dict(Sal=1, Sev=2, Maj2=3, Maj1=4, Mod=5, Min2=6, Min1=7, Typ=8),
  LandSlope=dict(Sev=1, Mod=2, Gtl=3),
  LotShape=dict(IR3=1, IR2=2, IR1=3, Reg=4),
  PavedDrive=dict(NA=0, N=1, P=2, Y=3),
  Street=dict(Grvl=1, Pave=2),
  Utilities=dict(ELO=1, NoSeWa=2, NoSewr=3, AllPub=4),
  BsmtCond=dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5),
)
# All the following variables can use the same replacement dictionary
quality_columns = [
  "BsmtCond",
  "BsmtQual",
  "ExterCond",
  "ExterQual",
  "FireplaceQu",
  "GarageCond",
  "GarageQual",
  "HeatingQC",
  "KitchenQual",
  "PoolQC",
]
quality_mapping = dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5)
for quality_column in quality_columns:
  replace_mapping[quality_column] = quality_mapping

all_X.replace(replace_mapping, inplace=True)

# Explicitly cast these variables to integers
ordinal_columns = list(replace_mapping.keys())
all_X[ordinal_columns] = all_X[ordinal_columns].astype(int)

### One hot encoding for nominal variables

We only have the nominal variables left.

- *Apply one-hot encoding with the [`pandas.get_dummies`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.get_dummies.html) function.*
- *How did the shape of `all_X` evolve?*

In [ ]:
# all_X = ???

#### Solution

In [ ]:
print(all_X.shape)
all_X = pandas.get_dummies(all_X)
print(all_X.shape)

In [ ]:
print("Columns:", ", ".join(all_X.columns))

Each category of each categorical variable was turned into a feature. The input matrix went from its original size of $\mathbb{R}^{2919 \times 79}$ to $\mathbb{R}^{2919 \times 244}$.

### Reassembling

Now that the cleaning process is over, we can split `all_X` back into `train_X` and `test_X`.

*Use the shape of `train_X` to know where to split `all_X`, then, perform the split.*

In [ ]:
# split_index = ???
# train_X = ???
# test_X = ???

#### Solution

In [ ]:
split_index = train_X.shape[0]
train_X = all_X.iloc[:split_index, :]
test_X = all_X.iloc[split_index:, :]

In [ ]:
train_X.shape

## Data exploration

### Target variable analysis

Let's start by analyzing the most important variable: the target variable. This [seaborn doc page](https://seaborn.pydata.org/tutorial/distributions.html#plotting-univariate-distributions) shows the different ways to explore such a variable.

*Use `seaborn` to visualize the output variable.*

In [ ]:
# Your code here

#### Solution

In [ ]:
seaborn.displot(train_y, kde=True)
plt.show()

### Comparing the Output Variable to a Normal Distribution

A prerequisite of many approaches is that the output variable is normally distributed.

*Test this hypothesis.*

In [ ]:
# Your code here

#### Solution

In [ ]:
seaborn.distplot(train_y, fit=scipy.stats.norm)
plt.show()

fig = plt.figure()
scipy.stats.probplot(train_y, dist="norm", plot=plt)
plt.show()

### Analyzing the Correlations to the Output Variable

Now that we have a good idea of the shape of the output variable, let's analyze the variables most correlated with it.

*Use [`DataFrame.corr`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corr.html) and [`seaborn.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html) to display a correlation matrix. Optionally, sort it by correlation with the output variable*

In [ ]:
# Your code here

#### Solution

In [ ]:
target_corrs = (
  train_X
  # Correlations with the target
  .corrwith(train_y)
  # Delete NaNs: they appear because of constant columns
  .dropna()
  # Sort by correlation with the target
  .sort_values(ascending=False)
)

# Get the top 10 corrs and anti-corrs
top_target_corrs = pandas.concat([target_corrs[:10], target_corrs[-10:]])
top_target_corrs.rename("SalePrice", inplace=True)

# Compute the correlation matrix
feature_corrs = train_X.loc[:, top_target_corrs.index].corr()

# Join target correlations with feature correlations
corrs = pandas.concat([top_target_corrs, feature_corrs], axis=1)

# Display with a heatmap
_, ax = plt.subplots(figsize=(10, 6))
seaborn.heatmap(corrs, vmin=-1, vmax=1, cmap=seaborn.diverging_palette(220, 20, n=100))
ax.set_title("Largest correlations and anti-correlations")
plt.show()

In [ ]:
corrs

### Exploring the Most Correlated Variables

We now know the most correlated variables. We still need to learn more about them.

*Plot the joint distribution of `GrLivArea` and `SalePrice`, as well as `OverallQual` and `SalePrice`.*

In [ ]:
# Your code here

#### Solution

In [ ]:
seaborn.jointplot(x=train_X["GrLivArea"], y=train_y)
plt.show()
seaborn.violinplot(x=train_X["OverallQual"], y=train_y)
plt.show()

### Analysis of the most correlated variables

*What do you notice on those two plots?*

#### Answer

We can see two outliers in the `GrLivArea` plot.

We need a deeper analysis to determine if those points should be kept or if they should be removed. How would you proceed?